Dependencies cell1

In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.2 MB/s eta 0:00:00


Hugging Face login cell2

In [2]:
from huggingface_hub import login
login()  # paste your HF token when prompted

Upload your JSONL files cell3

In [4]:
from google.colab import files
uploaded = files.upload()  # select train.jsonl and val.jsonl

Saving train.jsonl to train (1).jsonl
Saving val.jsonl to val (1).jsonl


Load dataset and format as chat prompts cell4

In [5]:
!pip uninstall -y pyarrow pandas datasets --quiet
!pip install pyarrow pandas datasets --quiet

import json
from datasets import Dataset
from transformers import AutoTokenizer # Import AutoTokenizer

# Define model_id for the tokenizer
model_id = "Qwen/Qwen3-4B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("train.jsonl")
val_data = load_jsonl("val.jsonl")

def format_example(ex):
    prompt = (
        f"<|system|>\nYou are a helpful {ex['role']} assistant at the company.\n"
        f"<|user|>\n{ex['instruction']}\n\nContext: {ex['context']}\n"
        f"<|assistant|>\n{ex['response']}{tokenizer.eos_token}"
    )
    return {"text": prompt}

train_ds = Dataset.from_list([format_example(e) for e in train_data])
val_ds = Dataset.from_list([format_example(e) for e in val_data])

print(train_ds[0]["text"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 102.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|system|>
You are a helpful HR assistant at the company.
<|user|>
Grievances case — a formal complaint names several events but provides limited supporting evidence

Context: Assess the case, establish the appropriate HR response, and coordinate the next step for the situation involving a formal complaint names several events but provides limited supporting evidence. Limited by statutory timelines for leave processing.
<|assistant|>
Facilitate a structured conversation focused on future working norms. Facilitate a structured conversation focused on future working norms; lighter-touch clarification, coaching or monitoring; formal HR or specialist review if policy or risk requires it. A facilitated agreement can repair collaboration without unnecessary formal escalation. Reviewed the relevant records, completed the mediation step, documented the rationale, and communicated the defined next action to the appropriate stakeholder(s).<|endoftext|>


unzipping adaptor


In [ ]:
!unzip -o slm_v1_adapter.zip

Archive:  slm_v1_adapter.zip
   creating: slm_v1_adapter/
  inflating: slm_v1_adapter/README.md  
  inflating: slm_v1_adapter/tokenizer_config.json  
  inflating: slm_v1_adapter/adapter_model.safetensors  
  inflating: slm_v1_adapter/tokenizer.json  
  inflating: slm_v1_adapter/adapter_config.json  
  inflating: slm_v1_adapter/chat_template.jinja  


Load Qwen3-4B-Base in 4-bit cell5

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen3-4B-Base"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

loading the saved model!!

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, "./slm_v1_adapter")
model.eval()
model.config.use_cache = True

ValueError: Can't find 'adapter_config.json' at './slm_v1_adapter'

Prepare model for k-bit training + attach LoRA cell6

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,796,480 || all params: 4,034,264,576 || trainable%: 0.2924


Tokenize the dataset cell7

In [8]:
MAX_LEN = 512

def tokenize_fn(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    # attention_mask is 1 for real tokens (including the true EOS), 0 for padding
    labels = [tok if mask == 1 else -100 for tok, mask in zip(input_ids, attention_mask)]
    tokens["labels"] = labels
    return tokens

train_tok = train_ds.map(tokenize_fn, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_fn, remove_columns=val_ds.column_names)

Map:   0%|          | 0/792 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [9]:
print(torch.cuda.memory_allocated()/1e9, "GB allocated")
print(torch.cuda.memory_reserved()/1e9, "GB reserved")

3.503304704 GB allocated
4.33061888 GB reserved


In [10]:
print(torch.cuda.is_available())
print(next(model.parameters()).device)
!nvidia-smi

True
cuda:0
Tue Aug 25 06:29:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             26W /   70W |    4251MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

Cell 8  Training arguments + Trainer

In [11]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./slm_v1_checkpoints",
    per_device_train_batch_size=1,  # Further reduced batch size to save memory
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 8 (1 * 8 = 8)
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    # Removed 'warmup_ratio' again as it caused a TypeError
    # warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # also fixes your warning
    dataloader_num_workers=2,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

In [12]:
import time
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(model.device) for k, v in batch.items()}
torch.cuda.synchronize()
t0 = time.time()
torch.cuda.empty_cache() # Added to clear any cached memory
out = model(**batch)
loss = out.loss
loss.backward()
torch.cuda.synchronize()
print("Single step time:", time.time() - t0, "seconds")

Single step time: 4.500471115112305 seconds


Cell 9 Train

In [13]:
trainer.train()

Step,Training Loss,Validation Loss
50,1.429293,1.485018
100,1.070111,0.979001
150,0.775773,0.744654
200,0.622615,0.630164
250,0.501066,0.590578
297,0.503594,0.584649


TrainOutput(global_step=297, training_loss=1.0342180568361121, metrics={'train_runtime': 15037.0231, 'train_samples_per_second': 0.158, 'train_steps_per_second': 0.02, 'total_flos': 2.660736859058995e+16, 'train_loss': 1.0342180568361121, 'epoch': 3.0})

Cell 10 Save the adapter

In [14]:
model.save_pretrained("./slm_v1_adapter")
tokenizer.save_pretrained("./slm_v1_adapter")

('./slm_v1_adapter/tokenizer_config.json',
 './slm_v1_adapter/chat_template.jinja',
 './slm_v1_adapter/tokenizer.json')

Cell 11  Zip and download it (so you don't lose it when the runtime resets)

In [15]:
!zip -r slm_v1_adapter.zip slm_v1_adapter
from google.colab import files
files.download("slm_v1_adapter.zip")

  adding: slm_v1_adapter/ (stored 0%)
  adding: slm_v1_adapter/tokenizer.json (deflated 81%)
  adding: slm_v1_adapter/README.md (deflated 65%)
  adding: slm_v1_adapter/adapter_config.json (deflated 60%)
  adding: slm_v1_adapter/chat_template.jinja (deflated 76%)
  adding: slm_v1_adapter/tokenizer_config.json (deflated 60%)
  adding: slm_v1_adapter/adapter_model.safetensors (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cell 12  Generate model responses on test set

In [17]:
import json
from tqdm import tqdm

# Load test data
test_data = load_jsonl("test.jsonl")

# Define build_prompt for inference
def build_prompt(ex):
    prompt = (
        f"<|system|>\nYou are a helpful {ex['role']} assistant at the company.\n"
        f"<|user|>\n{ex['instruction']}\n\nContext: {ex['context']}\n"
        f"<|assistant|>"
    )
    return prompt

results = []
for ex in tqdm(test_data):
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    results.append({
        "role": ex["role"],
        "instruction": ex["instruction"],
        "context": ex["context"],
        "expected_response": ex["response"],
        "generated_response": generated,
    })

with open("eval_generations.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Generated {len(results)} responses")
print(json.dumps(results[0], indent=2))

100%|██████████| 99/99 [41:41<00:00, 25.27s/it]

Generated 99 responses
{
  "role": "Product Manager",
  "instruction": "Retention Problems: test a recovery intervention Evaluate the underlying customer need, compare solution paths, and decide how the opportunity should affect the product plan.",
  "context": "test a recovery intervention target the point where churn risk becomes visible behavioral evidence supports intervention before cancellation Launch date has already been communicated externally; engineering is mid-implementation",
  "expected_response": "Customer impact first: test a recovery intervention. The customer pain is clear enough to justify action, but scope should stay tied to the demonstrated need. Reviewed the available evidence, compared the proposed solution with narrower and deferred alternatives, aligned the relevant stakeholders, and documented the roadmap decision using RICE. Faster resolution of the underlying issue and restored customer trust among long-tenured customers.",
  "generated_response": " Risk mi

In [ ]:
sample_text = train_ds[0]["text"]
print("--- Last 150 chars of formatted text ---")
print(repr(sample_text[-150:]))

tok_sample = tokenize_fn({"text": sample_text})
input_ids = tok_sample["input_ids"]
labels = tok_sample["labels"]

last_real_idx = max(i for i, l in enumerate(labels) if l != -100)
print("Last real (non-masked) token id:", input_ids[last_real_idx])
print("EOS token id:", tokenizer.eos_token_id)
print("Match?", input_ids[last_real_idx] == tokenizer.eos_token_id)

# Also check: how many real EOS tokens appear anywhere in input_ids (should be exactly 1, near the end of content)
eos_positions = [i for i, t in enumerate(input_ids) if t == tokenizer.eos_token_id]
print("All positions of EOS token id in this sequence:", eos_positions[:5], "... total:", len(eos_positions))

--- Last 150 chars of formatted text ---
'ords, completed the mediation step, documented the rationale, and communicated the defined next action to the appropriate stakeholder(s).<|endoftext|>'
Last real (non-masked) token id: 151643
EOS token id: 151643
Match? True
All positions of EOS token id in this sequence: [166, 167, 168, 169, 170] ... total: 346


In [21]:
ex = load_jsonl("test.jsonl")[0]
prompt = build_prompt(ex)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        # no repetition_penalty, no no_repeat_ngram_size this time
    )
n_generated = out.shape[1] - inputs["input_ids"].shape[1]
print("Tokens generated:", n_generated, "/ 200")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Tokens generated: 200 / 200
 Risk mitigation first: test a recovery intervention. Risk should influence scope rather than defeat it entirely, so improvements are kept only when the threat is validated. Reviewed the available evidence, compared the proposed solution with narrower and deferred alternatives, aligned the relevant stakeholders, and documented the roadmap decision using Value vs Effort. Faster resolution of the underlying issue and restored customer trust among new signups. Risk should influence scope rather than defeat it entirely, so improvements are kept only when the threat is validated. Revenue is one metric among many; unmeasured behavior can indicate real value. Limited impact on short-term delivery but creates a sustainable competitive advantage. Limited impact on short-term delivery but creates a sustainable competitive advantage. Limited scope change; short feedback loop on the identified risk. Limited scope change; short feedback loop on the identified risk. Limit

In [20]:

# Force-feed a known-good response and check EOS probability at the end
sample = load_jsonl("train.jsonl")[0]
full_text = f"<|system|>\nYou are a helpful {sample['role']} assistant at the company.\n<|user|>\n{sample['instruction']}\n\nContext: {sample['context']}\n<|assistant|>\n{sample['response']}"
inputs = tokenizer(full_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits

last_token_logits = logits[0, -1, :]
probs = torch.softmax(last_token_logits, dim=-1)
eos_prob = probs[tokenizer.eos_token_id].item()
top5 = torch.topk(probs, 5)
print("EOS probability at true end-of-response position:", eos_prob)
print("Top 5 predicted next tokens:", [(tokenizer.decode([t]), p.item()) for t, p in zip(top5.indices, top5.values)])

EOS probability at true end-of-response position: 3.750637134203316e-08
Top 5 predicted next tokens: [(' Cross', 0.2850744128227234), (' Training', 0.13465961813926697), (' Clear', 0.10487302392721176), (' Process', 0.06360869854688644), (' A', 0.043717578053474426)]


Cell 12b  Patch: truncate generations to a sane stopping point

In [19]:
import re

def smart_truncate(text, expected_len_tokens, max_extra_ratio=1.5):
    """
    Heuristic cleanup for a model with no learned EOS:
    1. Cut at first sign of drift: repeated fragments already handled by no_repeat_ngram,
       but we still cap length relative to expected response length.
    2. Trim to the last complete sentence within that cap (avoid ending mid-thought).
    """
    max_tokens = int(expected_len_tokens * max_extra_ratio)
    tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
        text = tokenizer.decode(tokens, skip_special_tokens=True)

    # trim to last full sentence so we don't cut mid-word/mid-clause
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sentences) > 1:
        text = " ".join(sentences[:-1]) if not sentences[-1].endswith((".", "!", "?")) else text

    return text.strip()

for r in results:
    expected_len = len(tokenizer(r["expected_response"], add_special_tokens=False)["input_ids"])
    r["generated_response_raw"] = r["generated_response"]
    r["generated_response"] = smart_truncate(r["generated_response"], expected_len)

with open("eval_generations.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results[0], indent=2))

{
  "role": "Product Manager",
  "instruction": "Retention Problems: test a recovery intervention Evaluate the underlying customer need, compare solution paths, and decide how the opportunity should affect the product plan.",
  "context": "test a recovery intervention target the point where churn risk becomes visible behavioral evidence supports intervention before cancellation Launch date has already been communicated externally; engineering is mid-implementation",
  "expected_response": "Customer impact first: test a recovery intervention. The customer pain is clear enough to justify action, but scope should stay tied to the demonstrated need. Reviewed the available evidence, compared the proposed solution with narrower and deferred alternatives, aligned the relevant stakeholders, and documented the roadmap decision using RICE. Faster resolution of the underlying issue and restored customer trust among long-tenured customers.",
  "generated_response": "Risk mitigation first: test an 

Cell 13 Automatic metrics

In [22]:
!pip install -q rouge-score

from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

ROLE_KEYWORDS = {
    "HR": ["policy", "employee", "grievance", "mediation", "conflict", "harassment", "leave", "escalat"],
    "Developer": ["bug", "code", "fix", "root cause", "test", "deploy", "refactor", "implementation"],
    "DevOps": ["incident", "infrastructure", "deploy", "monitor", "availability", "mitigat", "rollback"],
    "QA & Testing": ["test", "defect", "expected behavior", "actual behavior", "regression", "reproduce"],
    "Product Manager": ["roadmap", "stakeholder", "prioriti", "customer", "feature", "metric", "evidence"],
}

def role_adherence_score(role, text):
    keywords = ROLE_KEYWORDS.get(role, [])
    if not keywords:
        return None
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw.lower() in text_lower)
    return hits / len(keywords)

metrics_per_example = []
for r in results:
    rouge = scorer.score(r["expected_response"], r["generated_response"])
    exp_len = len(tokenizer(r["expected_response"], add_special_tokens=False)["input_ids"])
    gen_len = len(tokenizer(r["generated_response"], add_special_tokens=False)["input_ids"])

    metrics_per_example.append({
        "role": r["role"],
        "rouge1_f": rouge["rouge1"].fmeasure,
        "rouge2_f": rouge["rouge2"].fmeasure,
        "rougeL_f": rouge["rougeL"].fmeasure,
        "length_ratio": gen_len / exp_len if exp_len else None,
        "role_adherence": role_adherence_score(r["role"], r["generated_response"]),
    })

import pandas as pd
metrics_df = pd.DataFrame(metrics_per_example)

print("=== Overall averages ===")
print(metrics_df[["rouge1_f","rouge2_f","rougeL_f","length_ratio","role_adherence"]].mean())

print("\n=== Per-role averages ===")
print(metrics_df.groupby("role")[["rouge1_f","rouge2_f","rougeL_f","length_ratio","role_adherence"]].mean())

metrics_df.to_csv("layer3_automatic_metrics.csv", index=False)

=== Overall averages ===
rouge1_f          0.451171
rouge2_f          0.293942
rougeL_f          0.371838
length_ratio      1.343313
role_adherence    0.326599
dtype: float64

=== Per-role averages ===
                 rouge1_f  rouge2_f  rougeL_f  length_ratio  role_adherence
role                                                                       
DevOps           0.690418  0.629887  0.679669      1.297692        0.121429
Developer        0.290641  0.042687  0.153187      1.420261        0.269737
HR               0.379024  0.191510  0.262017      1.368517        0.143750
Product Manager  0.472007  0.347012  0.413373      1.403508        0.678571
QA & Testing     0.415740  0.246052  0.340011      1.230435        0.416667


Cell 14 Pull raw Developer + HR examples for inspection

In [23]:
def show_examples(role, n=3):
    subset = [r for r in results if r["role"] == role][:n]
    for i, r in enumerate(subset):
        print(f"--- {role} example {i+1} ---")
        print("INSTRUCTION:", r["instruction"])
        print()
        print("EXPECTED:", r["expected_response"])
        print()
        print("GENERATED:", r["generated_response"])
        print("\n" + "="*80 + "\n")

show_examples("Developer", n=3)
show_examples("HR", n=3)

--- Developer example 1 ---
INSTRUCTION: Stale replica reads Users sometimes see an old profile immediately after saving changes.

EXPECTED: I chose to route the short post-write consistency window to the primary using request-scoped state through the established boundary rather than add a one-off implementation. The approach solves the current problem while fitting an existing abstraction instead of adding another one-off path. The observed lag matches replica delay, so a global removal of replicas would be excessive. Implemented the change through the established module boundary and checked focused tests; recently updated profiles now read from the primary until the consistency window expires. Small maintenance costs compound, so avoiding a new special case can be as valuable as closing today's ticket. Read scaling should be balanced with explicit consistency requirements for user-visible workflows.

GENERATED: I chose to add read-side consistency checks and selectively invalidate st

Cell 15  Extract top vocabulary per role from expected responses (train + test) to build a data-driven keyword list

In [24]:
import re
from collections import Counter

STOPWORDS = set("""
the a an and or but if then to of in on for with as at by from this that
was were is are be been being it its it's their they them he she his her
i we you your our not no do does did done step next appropriate relevant
defined situation case action stakeholders rationale documented communicated
completed reviewed records
""".split())

def tokenize(text):
    return [w.lower() for w in re.findall(r"[a-zA-Z]+", text) if len(w) > 3 and w.lower() not in STOPWORDS]

all_train = load_jsonl("train.jsonl")  # re-uses train.jsonl if still uploaded, else re-upload it

role_vocab = {}
for role in set(r["role"] for r in all_train):
    texts = [r["response"] for r in all_train if r["role"] == role]
    words = []
    for t in texts:
        words.extend(tokenize(t))
    role_vocab[role] = Counter(words)

# Print top 20 per role
for role, counter in role_vocab.items():
    print(f"--- {role} ---")
    print(counter.most_common(20))
    print()

--- DevOps ---
[('health', 333), ('changes', 205), ('service', 201), ('recovery', 191), ('preserve', 182), ('signals', 182), ('known', 179), ('good', 179), ('path', 179), ('explicit', 174), ('keep', 159), ('intervention', 159), ('scoped', 159), ('logs', 159), ('guardrails', 159), ('available', 159), ('telemetry', 159), ('recent', 159), ('applied', 159), ('approach', 159)]

--- QA & Testing ---
[('evidence', 267), ('when', 174), ('observed', 173), ('executed', 160), ('scenario', 160), ('captured', 160), ('defect', 151), ('scope', 149), ('reproduction', 143), ('details', 140), ('retest', 128), ('coverage', 122), ('logged', 119), ('regression', 107), ('release', 102), ('risk', 87), ('test', 86), ('behavior', 78), ('than', 73), ('first', 67)]

--- Product Manager ---
[('roadmap', 217), ('evidence', 205), ('decision', 198), ('available', 160), ('compared', 160), ('proposed', 160), ('solution', 160), ('narrower', 160), ('deferred', 160), ('alternatives', 160), ('aligned', 160), ('using', 160

Cell 16  Rebuild role_adherence with data-driven keywords

In [25]:
ROLE_KEYWORDS = {
    "DevOps": ["health", "recovery", "signals", "guardrails", "telemetry", "logs", "scoped", "intervention", "service"],
    "QA & Testing": ["scenario", "defect", "reproduction", "retest", "coverage", "logged", "regression", "release"],
    "Product Manager": ["roadmap", "alternatives", "deferred", "narrower", "alignment", "customer", "metric", "primary"],
    "HR": ["policy", "coaching", "specialist", "formal", "monitoring", "lighter", "touch", "clarification", "stakeholder"],
    "Developer": ["boundary", "baseline", "implementation", "design", "chose", "failure", "path"],
}

def role_adherence_score(role, text):
    keywords = ROLE_KEYWORDS.get(role, [])
    if not keywords:
        return None
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw.lower() in text_lower)
    return hits / len(keywords)

# Recompute metrics with corrected keywords
metrics_per_example = []
for r in results:
    rouge = scorer.score(r["expected_response"], r["generated_response"])
    exp_len = len(tokenizer(r["expected_response"], add_special_tokens=False)["input_ids"])
    gen_len = len(tokenizer(r["generated_response"], add_special_tokens=False)["input_ids"])

    metrics_per_example.append({
        "role": r["role"],
        "rouge1_f": rouge["rouge1"].fmeasure,
        "rouge2_f": rouge["rouge2"].fmeasure,
        "rougeL_f": rouge["rougeL"].fmeasure,
        "length_ratio": gen_len / exp_len if exp_len else None,
        "role_adherence": role_adherence_score(r["role"], r["generated_response"]),
    })

metrics_df = pd.DataFrame(metrics_per_example)

print("=== Overall averages (corrected) ===")
print(metrics_df[["rouge1_f","rouge2_f","rougeL_f","length_ratio","role_adherence"]].mean())

print("\n=== Per-role averages (corrected) ===")
print(metrics_df.groupby("role")[["rouge1_f","rouge2_f","rougeL_f","length_ratio","role_adherence"]].mean())

metrics_df.to_csv("layer3_automatic_metrics_corrected.csv", index=False)

=== Overall averages (corrected) ===
rouge1_f          0.451171
rouge2_f          0.293942
rougeL_f          0.371838
length_ratio      1.343313
role_adherence    0.686147
dtype: float64

=== Per-role averages (corrected) ===
                 rouge1_f  rouge2_f  rougeL_f  length_ratio  role_adherence
role                                                                       
DevOps           0.690418  0.629887  0.679669      1.297692        0.972222
Developer        0.290641  0.042687  0.153187      1.420261        0.548872
HR               0.379024  0.191510  0.262017      1.368517        0.377778
Product Manager  0.472007  0.347012  0.413373      1.403508        0.793750
QA & Testing     0.415740  0.246052  0.340011      1.230435        0.731250


Cell 17 Latency measurement

In [26]:
import time

latencies = []
for ex in test_data[:20]:  # sample of 20 is enough, full 99 takes too long redundantly
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    n_new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    latencies.append({"total_s": elapsed, "tokens": n_new_tokens, "s_per_token": elapsed / n_new_tokens})

lat_df = pd.DataFrame(latencies)
print("Mean total latency (s):", lat_df["total_s"].mean())
print("Mean tokens generated:", lat_df["tokens"].mean())
print("Mean s/token:", lat_df["s_per_token"].mean())
print("P95 total latency (s):", lat_df["total_s"].quantile(0.95))

Mean total latency (s): 25.100215601921082
Mean tokens generated: 197.7
Mean s/token: 0.12703866759677984
P95 total latency (s): 25.629057097435


Cell 18 — Install embedding model + build the HR example index

In [27]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np
import json

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, good enough for this

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("train.jsonl")  # re-upload if not in this session
hr_examples = [r for r in train_data if r["role"] == "HR"]

hr_texts = [f"{ex['instruction']} {ex['context']}" for ex in hr_examples]
hr_embeddings = embedder.encode(hr_texts, convert_to_numpy=True, normalize_embeddings=True)

print(f"Indexed {len(hr_examples)} HR examples")
print("Embedding shape:", hr_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 160 HR examples
Embedding shape: (160, 384)


Cell 19 — Retrieval function

In [28]:
def retrieve_similar_hr_examples(query_instruction, query_context, k=2):
    query_text = f"{query_instruction} {query_context}"
    query_emb = embedder.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)
    sims = (hr_embeddings @ query_emb.T).flatten()  # cosine sim since normalized
    top_idx = np.argsort(-sims)[:k]
    return [hr_examples[i] for i in top_idx], sims[top_idx]

# quick test
test_hr = [r for r in load_jsonl("test.jsonl") if r["role"] == "HR"][0]
similar, scores = retrieve_similar_hr_examples(test_hr["instruction"], test_hr["context"], k=2)
for ex, score in zip(similar, scores):
    print(f"sim={score:.3f}")
    print("  instr:", ex["instruction"][:100])
    print("  approach hint:", ex["response"][:100])
    print()

sim=1.000
  instr: Leave/Policy Questions case — an employee requests leave using documentation that differs from the s
  approach hint: Use the formal framework with clear expectations, support and review dates. Use the formal framework

sim=1.000
  instr: Leave/Policy Questions case — an employee requests leave using documentation that differs from the s
  approach hint: Address the material risk first while preserving a fair process for the underlying case. Address the



Cell 20 — Build few-shot-augmented prompt and generate

In [29]:
from tqdm.auto import tqdm
import re

def smart_truncate(text, expected_len_tokens, max_extra_ratio=1.5):
    """
    Heuristic cleanup for a model with no learned EOS:
    1. Cut at first sign of drift: repeated fragments already handled by no_repeat_ngram,
       but we still cap length relative to expected response length.
    2. Trim to the last complete sentence within that cap (avoid ending mid-thought).
    """
    max_tokens = int(expected_len_tokens * max_extra_ratio)
    tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
        text = tokenizer.decode(tokens, skip_special_tokens=True)

    # trim to last full sentence so we don't cut mid-word/mid-clause
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sentences) > 1:
        text = " ".join(sentences[:-1]) if not sentences[-1].endswith((".", "!", "?")) else text

    return text.strip()

def build_fewshot_prompt(ex, k=2):
    similar, _ = retrieve_similar_hr_examples(ex["instruction"], ex["context"], k=k)

    fewshot_block = ""
    for s in similar:
        fewshot_block += (
            f"Example case:\n"
            f"Instruction: {s['instruction']}\n"
            f"Context: {s['context']}\n"
            f"Response: {s['response']}\n\n"
        )

    prompt = (
        f"<|system|>\nYou are a helpful HR assistant at the company. "
        f"Here are similar past cases for reference:\n\n{fewshot_block}"
        f"<|user|>\n{ex['instruction']}\n\nContext: {ex['context']}\n"
        f"<|assistant|>\n"
    )
    return prompt

# Generate on HR test subset with few-shot retrieval
hr_test = [r for r in load_jsonl("test.jsonl") if r["role"] == "HR"]

fewshot_results = []
for ex in tqdm(hr_test):
    prompt = build_fewshot_prompt(ex, k=2)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=900).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    fewshot_results.append({
        "role": "HR",
        "instruction": ex["instruction"],
        "context": ex["context"],
        "expected_response": ex["response"],
        "generated_response_raw": generated,
    })

# apply same smart_truncate patch
for r in fewshot_results:
    expected_len = len(tokenizer(r["expected_response"], add_special_tokens=False)["input_ids"])
    r["generated_response"] = smart_truncate(r["generated_response_raw"], expected_len)

with open("hr_fewshot_generations.json", "w") as f:
    json.dump(fewshot_results, f, indent=2)

print(json.dumps(fewshot_results[0], indent=2))

  0%|          | 0/20 [00:00<?, ?it/s]

{
  "role": "HR",
  "instruction": "Leave/Policy Questions case \u2014 an employee requests leave using documentation that differs from the standard format",
  "context": "Assess the case, establish the appropriate HR response, and coordinate the next step for the situation involving an employee requests leave using documentation that differs from the standard format. Budget constraints limit compensation adjustment options this cycle.",
  "expected_response": "Escalate the defined question to the appropriate specialist while preserving the case record. Escalate the defined question to the appropriate specialist while preserving the case record; lighter-touch clarification, coaching or monitoring; formal HR or specialist review if policy or risk requires it. Specialist review is appropriate because the consequences extend beyond routine HR discretion. Reviewed the relevant records, completed the escalation step, documented the rationale, and communicated the defined next action to the 

Cell 21 — Check the instruction/context mismatch

In [30]:
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

test_data_check = load_jsonl("test.jsonl")
hr_check = [r for r in test_data_check if r["role"] == "HR"]

# print first 3 HR test rows raw
for i, r in enumerate(hr_check[:3]):
    print(f"--- HR test row {i} ---")
    print("INSTRUCTION:", r["instruction"])
    print("CONTEXT:", r["context"])
    print()

--- HR test row 0 ---
INSTRUCTION: Leave/Policy Questions case — an employee requests leave using documentation that differs from the standard format
CONTEXT: Assess the case, establish the appropriate HR response, and coordinate the next step for the situation involving an employee requests leave using documentation that differs from the standard format. Budget constraints limit compensation adjustment options this cycle.

--- HR test row 1 ---
INSTRUCTION: Performance Management case — an employee's objectives are measurable but no longer match team priorities
CONTEXT: Assess the case, establish the appropriate HR response, and coordinate the next step for the situation involving an employee's objectives are measurable but no longer match team priorities. Must comply with company confidentiality standards regarding employee records.

--- HR test row 2 ---
INSTRUCTION: Manager-Employee Issues case — a manager and employee disagree about how priorities were communicated
CONTEXT: Assess

In [31]:
ROLE_KEYWORDS = {
    "DevOps": ["health", "recovery", "signals", "guardrails", "telemetry", "logs", "scoped", "intervention", "service"],
    "QA & Testing": ["scenario", "defect", "reproduction", "retest", "coverage", "logged", "regression", "release"],
    "Product Manager": ["roadmap", "alternatives", "deferred", "narrower", "alignment", "customer", "metric", "primary"],
    "HR": ["policy", "coaching", "specialist", "formal", "monitoring", "lighter", "touch", "clarification", "stakeholder"],
    "Developer": ["boundary", "baseline", "implementation", "design", "chose", "failure", "path"],
}

def role_adherence_score(role, text):
    keywords = ROLE_KEYWORDS.get(role, [])
    if not keywords:
        return None
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw.lower() in text_lower)
    return hits / len(keywords)

# Recompute metrics with corrected keywords
metrics_per_example = []
for r in results:
    rouge = scorer.score(r["expected_response"], r["generated_response"])
    exp_len = len(tokenizer(r["expected_response"], add_special_tokens=False)["input_ids"])
    gen_len = len(tokenizer(r["generated_response"], add_special_tokens=False)["input_ids"])

    metrics_per_example.append({
        "role": r["role"],
        "rouge1_f": rouge["rouge1"].fmeasure,
        "rouge2_f": rouge["rouge2"].fmeasure,
        "rougeL_f": rouge["rougeL"].fmeasure,
        "length_ratio": gen_len / exp_len if exp_len else None,
        "role_adherence": role_adherence_score(r["role"], r["generated_response"]),
    })

metrics_df = pd.DataFrame(metrics_per_example)

print("=== Overall averages (corrected) ===")
print(metrics_df[["rouge1_f","rouge2_f","rougeL_f","length_ratio","role_adherence"]].mean())

print("\n=== Per-role averages (corrected) ===")
print(metrics_df.groupby("role")[["rouge1_f","rouge2_f","rougeL_f","length_ratio","role_adherence"]].mean())

metrics_df.to_csv("layer3_automatic_metrics_corrected.csv", index=False)

=== Overall averages (corrected) ===
rouge1_f          0.451171
rouge2_f          0.293942
rougeL_f          0.371838
length_ratio      1.343313
role_adherence    0.686147
dtype: float64

=== Per-role averages (corrected) ===
                 rouge1_f  rouge2_f  rougeL_f  length_ratio  role_adherence
role                                                                       
DevOps           0.690418  0.629887  0.679669      1.297692        0.972222
Developer        0.290641  0.042687  0.153187      1.420261        0.548872
HR               0.379024  0.191510  0.262017      1.368517        0.377778
Product Manager  0.472007  0.347012  0.413373      1.403508        0.793750
QA & Testing     0.415740  0.246052  0.340011      1.230435        0.731250


In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio bitsandbytes peft transformers accelerate

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("PASTE_YOUR_OWN_NGROK_TOKEN_HERE")


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload slm_v1_adapter.zip
!unzip -o slm_v1_adapter.zip

Saving slm_v1_adapter.zip to slm_v1_adapter (3).zip
Archive:  slm_v1_adapter.zip
  inflating: slm_v1_adapter/tokenizer.json  
  inflating: slm_v1_adapter/README.md  
  inflating: slm_v1_adapter/adapter_config.json  
  inflating: slm_v1_adapter/chat_template.jinja  
  inflating: slm_v1_adapter/tokenizer_config.json  
  inflating: slm_v1_adapter/adapter_model.safetensors  


Load model on GPU (4-bit, same as training)

In [ ]:
!pip install --upgrade -q transformers peft


In [ ]:
!pip uninstall -y torchaudio torch torchvision
!pip install torch==2.9.0 torchaudio==2.9.0 torchvision==0.24.0 --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torch 2.13.0
Uninstalling torch-2.13.0:
  Successfully uninstalled torch-2.13.0
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0
Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 49.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 80.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 900.9/900.9 MB 874.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 9.0 MB/s eta 0:00:00
  Atte

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Ensure transformers and peft are compatible

ADAPTER_PATH = "./slm_v1_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Base",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
model.config.use_cache = True
print("Model loaded on GPU.")

ModuleNotFoundError: Could not import module 'BloomPreTrainedModel'. Are this object's requirements defined correctly?

Auth, rate limiting, generation logic (same as local version, inlined)

In [ ]:
import re
import time
from collections import defaultdict, deque

API_KEYS = {
    "dev-key-hr-001": {"employee_id": "E1001", "role": "HR"},
    "dev-key-dev-001": {"employee_id": "E1002", "role": "Developer"},
    "dev-key-pm-001": {"employee_id": "E1003", "role": "Product Manager"},
    "dev-key-devops-001": {"employee_id": "E1004", "role": "DevOps"},
    "dev-key-qa-001": {"employee_id": "E1005", "role": "QA & Testing"},
}

_request_log = defaultdict(deque)
MAX_REQUESTS = 10
WINDOW_SECONDS = 60

def check_rate_limit(key):
    now = time.time()
    log = _request_log[key]
    while log and now - log[0] > WINDOW_SECONDS:
        log.popleft()
    if len(log) >= MAX_REQUESTS:
        return False
    log.append(now)
    return True

def build_prompt(role, instruction, context):
    return (
        f"<|system|>\nYou are a helpful {role} assistant at the company.\n"
        f"<|user|>\n{instruction}\n\nContext: {context}\n"
        f"<|assistant|>\n"
    )

def smart_truncate(text, target_len_tokens=90, max_extra_ratio=1.5):
    max_tokens = int(target_len_tokens * max_extra_ratio)
    tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
        text = tokenizer.decode(tokens, skip_special_tokens=True)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sentences) > 1 and not sentences[-1].endswith( (".", "!", "?") ):
        text = " ".join(sentences[:-1])
    return text.strip()

def generate_response(role, instruction, context):
    prompt = build_prompt(role, instruction, context)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=20, # Further reduced max_new_tokens to mitigate potential OOM issues
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return smart_truncate(raw)

FastAPI app

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import threading
import logging # Import logging

nest_asyncio.apply()

# Configure logging to see messages in Colab output
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

app = FastAPI(title="SLM Backend - Colab")

class GenerateRequest(BaseModel):
    instruction: str
    context: str = ""

class GenerateResponse(BaseModel):
    role: str
    response: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest, x_api_key: str = Header(...)):
    logger.info(f"Received /generate request. Instruction: '{req.instruction[:50]}...', Context: '{req.context[:50]}...' ")
    record = API_KEYS.get(x_api_key)
    if record is None:
        logger.warning(f"Invalid API key: {x_api_key}")
        raise HTTPException(status_code=401, detail="Invalid API key.")

    logger.info(f"API Key valid for employee_id: {record['employee_id']}, role: {record['role']}")

    if not check_rate_limit(record["employee_id"]):
        logger.warning(f"Rate limit exceeded for employee_id: {record['employee_id']}")
        raise HTTPException(status_code=429, detail="Rate limit exceeded.")
    if not req.instruction.strip():
        logger.error("Instruction cannot be empty.")
        raise HTTPException(status_code=400, detail="instruction cannot be empty.")

    logger.info(f"Calling generate_response for role: {record['role']}")
    response_text = generate_response(record["role"], req.instruction, req.context)
    logger.info(f"generate_response returned text: '{response_text[:50]}'...")

    final_response = GenerateResponse(role=record["role"], response=response_text)
    logger.info("Returning successful response.")
    return final_response

Start the server + expose via ngrok

In [ ]:
import time
import threading
import uvicorn
from pyngrok import ngrok
import nest_asyncio
import socket
import requests # Import requests for health check

nest_asyncio.apply()

# Global variables (reset each time the cell is run to ensure a fresh start)
_uvicorn_server_thread = None
_ngrok_public_url_obj = None
_uvicorn_port = 8000 # Default starting port

def _find_free_port(start_port=_uvicorn_port, max_tries=10):
    for i in range(max_tries):
        port = start_port + i
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('localhost', port))
                return port
            except OSError:
                continue # Try next port
    raise RuntimeError("Could not find a free port.")

def _run_uvicorn_server(port):
    # Ensure app is accessible in this scope if it's not truly global or dynamically loaded
    # In Colab, top-level defined 'app' usually is global.
    config = uvicorn.Config(app, host="0.0.0.0", port=port, log_level="info")
    server = uvicorn.Server(config)
    server.run()

# --- Start of cell execution logic ---

# 1. Kill any existing ngrok tunnels to free up processes associated with ngrok
try:
    ngrok.kill()
    print("Killed any lingering ngrok processes and tunnels.")
except Exception as e:
    print(f"Warning: Could not kill ngrok processes: {e}")

# Find a free port for Uvicorn
global _uvicorn_port # Ensure we can modify the global default port
_uvicorn_port = _find_free_port()
print(f"Found free port: {_uvicorn_port}")

# Start a new ngrok tunnel
public_url = ngrok.connect(_uvicorn_port)
_ngrok_public_url_obj = public_url # Store for potential reuse in the same execution cycle or debug
print("Public URL:", public_url)

# Start Uvicorn server in a new thread
_uvicorn_server_thread = threading.Thread(target=_run_uvicorn_server, args=(_uvicorn_port,), daemon=True)
_uvicorn_server_thread.start()
print("INFO: Started new Uvicorn server thread on port", _uvicorn_port)

# Wait for the server to be ready with a health check
health_url = f"{public_url.public_url}/health"
max_retries = 30

# Add a small initial sleep to allow Uvicorn thread to start binding
time.sleep(2)

server_ready = False
for i in range(max_retries):
    try:
        response = requests.get(health_url, timeout=1)
        if response.status_code == 200:
            print("INFO: Server is ready!")
            server_ready = True
            break
    except requests.exceptions.RequestException as e:
        pass # Keep silent unless debugging
    time.sleep(1)

if not server_ready:
    print("ERROR: Server did not become ready within the timeout.")



Killed any lingering ngrok processes and tunnels.
Found free port: 8000


INFO:     Started server process [13802]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://mongoose-glaring-bottom.ngrok-free.dev" -> "http://localhost:8000"
INFO: Started new Uvicorn server thread on port 8000
INFO:     34.186.21.118:0 - "GET /health HTTP/1.1" 200 OK
INFO: Server is ready!


Test it (from within Colab, or use the printed public URL anywhere)

In [ ]:
import requests

resp = requests.post(
    f"{public_url.public_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Employee requests leave with non-standard documentation", "context": "Policy allows exceptions with manager approval."},
)
print(resp.status_code)
print(resp.json())

INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, i

500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Test: send 12 rapid requests

In [ ]:
import requests

base_url = public_url.public_url

for i in range(12):
    resp = requests.post(
        f"{base_url}/generate",
        headers={"X-API-Key": "dev-key-hr-001"},
        json={"instruction": "Quick test request", "context": "test"},
    )
    print(f"Request {i+1}: status={resp.status_code}")
    if resp.status_code == 429:
        print("  -> Rate limited:", resp.json())

INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 1: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 2: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 3: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 4: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 5: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 6: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 7: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 8: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 9: status=200
INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 200 OK
Request 10: status=200


INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
Request 11: status=429
  -> Rate limited: {'detail': 'Rate limit exceeded.'}


INFO:     34.186.21.118:0 - "POST /generate HTTP/1.1" 429 Too Many Requests
Request 12: status=429
  -> Rate limited: {'detail': 'Rate limit exceeded.'}


checking env.

In [ ]:
!python -c "import torch; print('Torch version:', torch.__version__); print('Torch CUDA version:', torch.version.cuda)"
!nvcc --version

Torch version: 2.13.0+cu130
Torch CUDA version: 13.0
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


Reinstall your original Layer 4 dependencies (no vLLM this time)

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio bitsandbytes peft transformers accelerate

Set ngrok token again (fresh runtime = lost previous session state)

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("PASTE_YOUR_OWN_NGROK_TOKEN_HERE")


In [ ]:
from google.colab import files
uploaded = files.upload()  # slm_v1_adapter.zip
!unzip -o slm_v1_adapter.zip

Saving slm_v1_adapter.zip to slm_v1_adapter (1).zip
Archive:  slm_v1_adapter.zip
  inflating: slm_v1_adapter/tokenizer.json  
  inflating: slm_v1_adapter/README.md  
  inflating: slm_v1_adapter/adapter_config.json  
  inflating: slm_v1_adapter/chat_template.jinja  
  inflating: slm_v1_adapter/tokenizer_config.json  
  inflating: slm_v1_adapter/adapter_model.safetensors  


sqllite

In [ ]:
import sqlite3
import json
from datetime import datetime
from collections import defaultdict

DB_PATH = "./employee_profiles.db"

def _get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with _get_conn() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS profiles (
                employee_id     TEXT PRIMARY KEY,
                role            TEXT NOT NULL,
                response_style  TEXT DEFAULT 'balanced',
                verbosity       TEXT DEFAULT 'medium',
                language        TEXT DEFAULT 'english',
                focus_areas     TEXT DEFAULT '[]',
                updated_at      TEXT
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS memories (
                id              INTEGER PRIMARY KEY AUTOINCREMENT,
                employee_id     TEXT NOT NULL,
                summary         TEXT NOT NULL,
                approved        INTEGER DEFAULT 1,
                created_at      TEXT,
                FOREIGN KEY(employee_id) REFERENCES profiles(employee_id)
            )
        """)
        conn.commit()
    print("Profile DB initialized.")

def upsert_profile(employee_id, role, preferences=None):
    prefs = preferences or {}
    with _get_conn() as conn:
        conn.execute("""
            INSERT INTO profiles (employee_id, role, response_style, verbosity, language, focus_areas, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(employee_id) DO UPDATE SET
                role=excluded.role, response_style=excluded.response_style,
                verbosity=excluded.verbosity, language=excluded.language,
                focus_areas=excluded.focus_areas, updated_at=excluded.updated_at
        """, (employee_id, role,
              prefs.get("response_style", "balanced"),
              prefs.get("verbosity", "medium"),
              prefs.get("language", "english"),
              json.dumps(prefs.get("focus_areas", [])),
              datetime.utcnow().isoformat()))
        conn.commit()

def get_profile(employee_id):
    with _get_conn() as conn:
        row = conn.execute("SELECT * FROM profiles WHERE employee_id = ?", (employee_id,)).fetchone()
    if row is None:
        return None
    d = dict(row)
    d["focus_areas"] = json.loads(d["focus_areas"])
    return d

def add_memory(employee_id, summary, approved=True):
    with _get_conn() as conn:
        conn.execute("INSERT INTO memories (employee_id, summary, approved, created_at) VALUES (?, ?, ?, ?)",
                     (employee_id, summary, int(approved), datetime.utcnow().isoformat()))
        conn.commit()

def get_approved_memories(employee_id, limit=3):
    with _get_conn() as conn:
        rows = conn.execute("""SELECT summary FROM memories
            WHERE employee_id = ? AND approved = 1
            ORDER BY created_at DESC LIMIT ?""", (employee_id, limit)).fetchall()
    return [r["summary"] for r in rows]

def list_memories(employee_id):
    with _get_conn() as conn:
        rows = conn.execute("""SELECT id, summary, approved, created_at FROM memories
            WHERE employee_id = ? ORDER BY created_at DESC""", (employee_id,)).fetchall()
    return [dict(r) for r in rows]

def delete_memory(memory_id):
    with _get_conn() as conn:
        conn.execute("DELETE FROM memories WHERE id = ?", (memory_id,))
        conn.commit()

init_db()

Profile DB initialized.


context builder

In [ ]:
VERBOSITY_MAP = {
    "brief":    "Keep your response concise — 2-3 sentences maximum.",
    "medium":   "Give a clear, complete response without unnecessary elaboration.",
    "detailed": "Provide a thorough, detailed response with full reasoning.",
}
STYLE_MAP = {
    "balanced":   "Balance directness with nuance.",
    "direct":     "Be direct and action-oriented. Lead with the recommendation.",
    "analytical": "Prioritize analysis and reasoning over direct recommendations.",
    "empathetic": "Prioritize understanding and tone before moving to action.",
}

def build_context(employee_id, role, instruction, context):
    profile = get_profile(employee_id)
    if profile is None:
        return {
            "system_prefix": f"You are a helpful {role} assistant at the company.",
            "instruction": instruction, "context": context, "profile_found": False,
        }
    verbosity_instr = VERBOSITY_MAP.get(profile["verbosity"], VERBOSITY_MAP["medium"])
    style_instr = STYLE_MAP.get(profile["response_style"], STYLE_MAP["balanced"])
    focus_areas = profile.get("focus_areas", [])
    focus_instr = f"\nThe employee's focus areas: {', '.join(focus_areas)}." if focus_areas else ""
    system_prefix = f"You are a helpful {role} assistant at the company.\n{style_instr} {verbosity_instr}{focus_instr}"
    memories = get_approved_memories(employee_id, limit=3)
    enriched_context = context
    if memories:
        memory_block = "\n".join(f"- {m}" for m in memories)
        enriched_context = f"[Past context:]\n{memory_block}\n\n{context}"
    return {
        "system_prefix": system_prefix, "instruction": instruction,
        "context": enriched_context, "profile_found": True,
    }

def build_prompt_from_context(ctx):
    return (
        f"<|system|>\n{ctx['system_prefix']}\n"
        f"<|user|>\n{ctx['instruction']}\n\nContext: {ctx['context']}\n"
        f"<|assistant|>\n"
    )

print("Context builder ready.")

Context builder ready.


In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import threading
import logging

nest_asyncio.apply()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="SLM Backend - Layer 4+5")

class GenerateRequest(BaseModel):
    instruction: str
    context: str = ""

class GenerateResponse(BaseModel):
    role: str
    response: str
    personalized: bool

class PreferencesModel(BaseModel):
    response_style: Optional[str] = "balanced"
    verbosity: Optional[str] = "medium"
    language: Optional[str] = "english"
    focus_areas: Optional[List[str]] = []

class ProfileRequest(BaseModel):
    preferences: PreferencesModel = PreferencesModel()

class MemoryRequest(BaseModel):
    summary: str
    approved: bool = True

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest, x_api_key: str = Header(...)):
    record = API_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="Invalid API key.")
    if not check_rate_limit(record["employee_id"]):
        raise HTTPException(status_code=429, detail="Rate limit exceeded.")
    if not req.instruction.strip():
        raise HTTPException(status_code=400, detail="instruction cannot be empty.")

    # Layer 5: build personalized context
    ctx = build_context(record["employee_id"], record["role"], req.instruction, req.context)
    prompt = build_prompt_from_context(ctx)

    # generate from enriched prompt
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=600).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            repetition_penalty=1.3, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    response_text = smart_truncate(raw)

    return GenerateResponse(role=record["role"], response=response_text, personalized=ctx["profile_found"])

@app.put("/profile")
def update_profile(req: ProfileRequest, x_api_key: str = Header(...)):
    record = API_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="Invalid API key.")
    upsert_profile(record["employee_id"], record["role"], req.preferences.model_dump())
    return {"message": "Profile updated.", "employee_id": record["employee_id"]}

@app.get("/profile")
def fetch_profile(x_api_key: str = Header(...)):
    record = API_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="Invalid API key.")
    profile = get_profile(record["employee_id"])
    return profile or {"message": "No profile yet.", "employee_id": record["employee_id"]}

@app.post("/memory")
def create_memory(req: MemoryRequest, x_api_key: str = Header(...)):
    record = API_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="Invalid API key.")
    if not req.summary.strip():
        raise HTTPException(status_code=400, detail="Memory summary cannot be empty.")
    add_memory(record["employee_id"], req.summary, req.approved)
    return {"message": "Memory saved."}

@app.get("/memories")
def fetch_memories(x_api_key: str = Header(...)):
    record = API_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="Invalid API key.")
    return {"memories": list_memories(record["employee_id"])}

@app.delete("/memory/{memory_id}")
def remove_memory(memory_id: int, x_api_key: str = Header(...)):
    record = API_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="Invalid API key.")
    delete_memory(memory_id)
    return {"message": f"Memory {memory_id} deleted."}

print("Layer 4+5 app defined.")

Layer 4+5 app defined.


In [ ]:
base_url = public_url.public_url

# 1. Set HR employee profile
requests.put(f"{base_url}/profile",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"preferences": {"response_style": "direct", "verbosity": "brief", "focus_areas": ["leave management", "grievances"]}})

# 2. Add a memory
requests.post(f"{base_url}/memory",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"summary": "This employee prefers escalation to specialists for policy-grey-area cases."})

# 3. Generate — personalized=True this time
resp = requests.post(f"{base_url}/generate",
    headers={"X-API-Key": "dev-key-hr-001"},
    json={"instruction": "Employee requests leave with non-standard documentation",
          "context": "Manager is unavailable for approval."})
print(resp.json())

# 4. Fetch profile to confirm it saved
print(requests.get(f"{base_url}/profile", headers={"X-API-Key": "dev-key-hr-001"}).json())

# 5. Fetch memories
print(requests.get(f"{base_url}/memories", headers={"X-API-Key": "dev-key-hr-001"}).json())

NameError: name 'public_url' is not defined